# 无参考评估：NIQE / MANIQA / CLIPIQA

本 notebook 对 **已有 SR 推理结果** 做无 GT 质量评估，不重新跑推理。

- **NIQE**：自然图像质量评估，越低越好（自然度越高）
- **MANIQA**：多维度图像质量，越高越好
- **CLIPIQA**：基于 CLIP 的感知质量，越高越好

使用 `dataset/test_list.txt` 中的测试集（434 张），仅对列表内图像计分。

In [1]:
# ============================================================
# 0. 配置
# ============================================================

import os
import glob
import json
import numpy as np
from tqdm import tqdm

PROJECT_DIR = os.getcwd()
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# 测试集名单（共 434 张）
TEST_LIST_PATH = os.path.join(PROJECT_DIR, 'dataset/test_list.txt')

# 待评估的 SR 图像目录（直接使用全参考评估脚本生成的推理结果）
SR_DIR = os.path.join(PROJECT_DIR, 'results/GRL_2x_finetune_fullref_sr')

# 无参考评估结果保存名（与 SR 目录对应）
EVAL_NAME = 'GRL_2x_finetune'

print('PROJECT_DIR:', PROJECT_DIR)
print('SR_DIR:', SR_DIR)
print('TEST_LIST:', TEST_LIST_PATH)

PROJECT_DIR: d:\lixinqi\UCSD\ECE285\AWM
SR_DIR: d:\lixinqi\UCSD\ECE285\AWM\results/GRL_2x_finetune_fullref_sr
TEST_LIST: d:\lixinqi\UCSD\ECE285\AWM\dataset/test_list.txt


In [2]:
# ============================================================
# 1. 加载测试集名单与 SR 路径
# ============================================================

with open(TEST_LIST_PATH, 'r', encoding='utf-8') as f:
    test_set = set(line.strip() for line in f if line.strip())

all_sr_paths = sorted(glob.glob(os.path.join(SR_DIR, '*.[jp][pn]*g')))
sr_paths = [p for p in all_sr_paths if os.path.basename(p) in test_set]

print(f'测试集名单: {len(test_set)} 张')
print(f'SR 目录内总图像: {len(all_sr_paths)} 张')
print(f'本次评估（测试集内）: {len(sr_paths)} 张')

if len(sr_paths) == 0:
    raise FileNotFoundError(f'SR_DIR 中无测试集内图像，请检查 {SR_DIR} 与 {TEST_LIST_PATH}')

测试集名单: 434 张
SR 目录内总图像: 434 张
本次评估（测试集内）: 434 张


In [3]:
# ============================================================
# 2. 初始化无参考指标（NIQE / MANIQA / CLIPIQA）
# ============================================================

import torch
import pyiqa

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

print('初始化 pyiqa 指标（首次运行会下载权重）...')
niqe_metric = pyiqa.create_metric('niqe', device=device)
maniqa_metric = pyiqa.create_metric('maniqa', device=device)
clipiqa_metric = pyiqa.create_metric('clipiqa', device=device)
print('OK.')

Device: cuda
初始化 pyiqa 指标（首次运行会下载权重）...


c:\Users\admin\.conda\envs\ece284fa25\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading pretrained model MANIQA from C:\Users\admin\.cache\torch\hub\pyiqa\ckpt_koniq10k.pt
OK.


In [4]:
# ============================================================
# 3. 逐图计算 NIQE / MANIQA / CLIPIQA 并汇总
# ============================================================

niqe_scores, maniqa_scores, clipiqa_scores = [], [], []
per_image = {}

for path in tqdm(sr_paths, desc='No-ref eval'):
    fname = os.path.basename(path)
    n = niqe_metric(path).item()
    m = maniqa_metric(path).item()
    c = clipiqa_metric(path).item()
    niqe_scores.append(n)
    maniqa_scores.append(m)
    clipiqa_scores.append(c)
    per_image[fname] = {'NIQE': round(n, 4), 'MANIQA': round(m, 4), 'CLIPIQA': round(c, 4)}

n_img = len(niqe_scores)
avg_niqe = float(np.mean(niqe_scores))
avg_maniqa = float(np.mean(maniqa_scores))
avg_clipiqa = float(np.mean(clipiqa_scores))

print('\n' + '='*50)
print(f'  {EVAL_NAME} 无参考评估（测试集 {n_img} 张）')
print('='*50)
print(f'  NIQE    (↓ 越低越好): {avg_niqe:.4f}')
print(f'  MANIQA  (↑ 越高越好): {avg_maniqa:.4f}')
print(f'  CLIPIQA (↑ 越高越好): {avg_clipiqa:.4f}')
print('='*50)

No-ref eval: 100%|██████████| 434/434 [17:09<00:00,  2.37s/it]  


  GRL_2x_finetune 无参考评估（测试集 434 张）
  NIQE    (↓ 越低越好): 5.3640
  MANIQA  (↑ 越高越好): 0.3243
  CLIPIQA (↑ 越高越好): 0.5541


In [5]:
# ============================================================
# 4. 保存 JSON 与 TXT
# ============================================================

summary = {
    'model': EVAL_NAME,
    'sr_dir': SR_DIR,
    'test_list': TEST_LIST_PATH,
    'num_images': n_img,
    'metrics': {
        'NIQE': round(avg_niqe, 4),
        'MANIQA': round(avg_maniqa, 4),
        'CLIPIQA': round(avg_clipiqa, 4),
    },
    'per_image': per_image,
}

json_path = os.path.join(RESULTS_DIR, f'{EVAL_NAME}_noref_eval.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f'已保存 JSON: {json_path}')

txt_path = os.path.join(RESULTS_DIR, f'{EVAL_NAME}_noref_eval.txt')
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(f'{EVAL_NAME} (无参考, 测试集 {n_img} 张)\n')
    f.write('='*40 + '\n')
    f.write(f'NIQE    (↓ 越低越好): {avg_niqe:.4f}\n')
    f.write(f'MANIQA  (↑ 越高越好): {avg_maniqa:.4f}\n')
    f.write(f'CLIPIQA (↑ 越高越好): {avg_clipiqa:.4f}\n')
    f.write('='*40 + '\n')
print(f'已保存 TXT:  {txt_path}')
print('\n✅ 无参考评估完成。')

已保存 JSON: d:\lixinqi\UCSD\ECE285\AWM\results\GRL_2x_finetune_noref_eval.json
已保存 TXT:  d:\lixinqi\UCSD\ECE285\AWM\results\GRL_2x_finetune_noref_eval.txt

✅ 无参考评估完成。
